# 07 - Grad-CAM by prediction category

Grad-CAM analysis of the run chosen with `evaluation.run` (parameters in the `explanation` section of
`configs/default.yaml`). Every validation record falls in one category:
- MC dropout prediction **right** or **wrong**;
- uncertainty (`explanation.uncertainty_measure`) **low** (below `low_quantile`), **high** (above `high_quantile`) or
  **middle**;
- true class **good** or **bad**.

Two uses of the maps:
1. **aggregate**, over all validation records: the map of the predicted class (MC mean, normalized to unit sum) is
   reduced to its profile over time and frequency, the share on zero padding, the share before the anchor and its
   instability across MC passes; these are compared between categories, and the mean map of every category is drawn;
2. **single records**: `records_per_category` records drawn at random from each low- and high-uncertainty category,
   with the maps of both classes over the image and the waveform (figures for presentations and papers).

Outputs in `<run>/explanation/`. The Grad-CAM step takes `mc_samples` x 2 passes per record: on the full validation
set it takes several minutes.

## 0. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from wqu.config import load_config
from wqu.data.loading import CV_COMPONENTS
from wqu.data.waveforms import iter_waveform_batches, load_waveforms, stored_sampling_rate_hz
from wqu.evaluation.explanations import (
    CATEGORY_COLUMN,
    EXTREME_LEVELS,
    aggregate_grad_cam,
    assign_categories,
    category_counts,
    category_order,
    map_summary,
    mean_profiles_by_category,
    padding_profile,
    sample_category_records,
    summary_by_category,
)
from wqu.evaluation.uncertainty_assessment import MEASURE_PREDICTIONS
from wqu.models.cred import build_model
from wqu.models.explain import grad_cam_by_class
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    feature_file_path,
    grid_from_config,
    process_record,
    read_feature_records,
    read_features,
)
from wqu.training.data import FeatureDataset, FrequencyStatistics, make_loader
from wqu.training.trainer import BEST_MODEL_FILE, VALIDATION_PREDICTIONS_FILE, find_run_dir, select_device
from wqu.visualization import (
    apply_plot_style,
    plot_category_distributions,
    plot_category_mean_maps,
    plot_category_profiles,
    plot_grad_cam,
    save_figure,
    save_figures_pdf,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

# The traces are read from the same source as the features of the run (dev subset or full SeisBench datasets).
config = load_config(CONFIG_PATH)
config = replace(config, waveforms=replace(config.waveforms, source=config.training.feature_source))
columns = config.columns
training = config.training
explanation = config.explanation
preprocessing = config.preprocessing
apply_plot_style()
device = select_device(training.device)

run_dir = find_run_dir(training.runs_dir, training.representation, config.evaluation.run)
output_dir = run_dir / "explanation"
output_dir.mkdir(exist_ok=True)
representation = run_dir.name.split("_")[0]
predictions = pd.read_csv(run_dir / VALIDATION_PREDICTIONS_FILE)

features_path = feature_file_path(preprocessing, training.feature_source)
records = read_feature_records(features_path, columns.text)
validation_records = records.loc[predictions["record"].to_numpy()]
statistics = FrequencyStatistics.load(run_dir / "normalization.npz")
model = build_model(config.model, len(CV_COMPONENTS), len(statistics.mean))
model.load_state_dict(torch.load(run_dir / BEST_MODEL_FILE, map_location="cpu"))
_, grid = read_features(features_path, representation, validation_records[FEATURE_INDEX_COLUMN].to_numpy()[:1])
print(f"Run: {run_dir.name} ({representation}); validation records: {len(predictions):,}; device: {device}")

## 1. Categories

Low and high uncertainty are the lowest and highest `low_quantile` / `1 - high_quantile` of the measure over the
validation set. Rows: uncertainty level; columns: correctness of the MC prediction and true class.

In [ ]:
categories = assign_categories(predictions, columns.label, explanation.uncertainty_measure,
                               explanation.low_quantile, explanation.high_quantile)
categories.to_csv(output_dir / "categories.csv", index=False)
counts = category_counts(categories)
counts.to_csv(output_dir / "category_counts.csv")
counts

## 2. Grad-CAM maps of all validation records

Map of the predicted class for every record (MC mean over `explanation.mc_samples` passes), reduced to time and
frequency profiles and to the instability across passes; the mean map of each category is accumulated.

In [ ]:
dataset = FeatureDataset(features_path, representation, validation_records[FEATURE_INDEX_COLUMN].to_numpy(),
                         validation_records[columns.label].to_numpy(), statistics)
loader = make_loader(dataset, explanation.batch_size, False, training.num_workers, training.random_seed)
start = time.perf_counter()
aggregate = aggregate_grad_cam(model, model.last_conv_layer, loader, categories,
                               MEASURE_PREDICTIONS[explanation.uncertainty_measure], explanation.mc_samples, device)
print(f"Grad-CAM maps of {len(categories):,} records in {(time.perf_counter() - start) / 60:.1f} min")

## 3. Zero padding in each window

Share of zero-padded samples (mean over components) in each time bin of the window, from the stored traces and the
window bounds saved by the preprocessing.

In [ ]:
sampling_rate_hz = stored_sampling_rate_hz(records)
profiles_by_record = {}
for batch, traces in iter_waveform_batches(validation_records, config):
    for (index, record), trace in zip(batch.iterrows(), traces):
        profiles_by_record[index] = padding_profile(trace, int(record["window_start"]), int(record["window_stop"]),
                                                    sampling_rate_hz, grid.times_s, config.waveforms.zero_tolerance)
padding_profiles = np.stack([profiles_by_record[index] for index in validation_records.index])
print(f"Mean share of padding in the window: {padding_profiles.mean():.3f}")

## 4. Map quantities by category

Per record: share of the map on zero padding (`padding_share`; compare with `window_padding_share`, the share of the
window that is padding: a uniform map would give the same value), share before the anchor (`pre_anchor_share`; the
window has `pre_anchor_s` of `window_length_s` before the anchor) and instability (mean MC standard deviation over mean
of the map). Medians per category, then distributions (boxes without outliers; dashed boxes: true class bad).

In [ ]:
summary = map_summary(aggregate, padding_profiles, grid.times_s, preprocessing.pre_anchor_s)
summary.insert(0, "record", categories["record"].to_numpy())
summary.insert(1, CATEGORY_COLUMN, categories[CATEGORY_COLUMN].to_numpy())
summary.to_csv(output_dir / "map_summary.csv", index=False)
by_category = summary_by_category(summary.drop(columns=["record", CATEGORY_COLUMN]), summary[CATEGORY_COLUMN])
by_category.to_csv(output_dir / "map_summary_by_category.csv")
print(f"Uniform map: pre_anchor_share = {preprocessing.pre_anchor_s / preprocessing.window_length_s:.3f}")
by_category.round(4)

In [ ]:
QUANTITIES = {
    "padding_share": "share of the map on padding",
    "pre_anchor_share": "share of the map before the anchor",
    "instability": "MC instability of the map",
}
figure, axes = plt.subplots(1, len(QUANTITIES), figsize=(17, 5), constrained_layout=True)
for ax, (column, label) in zip(axes, QUANTITIES.items()):
    plot_category_distributions(summary[column], summary[CATEGORY_COLUMN], category_order(), label, ax)
figure.suptitle(f"Grad-CAM quantities by category: {run_dir.name}", x=0.01, ha="left")
save_figure(figure, output_dir / "map_quantities_by_category.png")
save_figure(figure, output_dir / "map_quantities_by_category.pdf")

## 5. Mean maps and profiles of the low- and high-uncertainty categories

Mean normalized map of each category on a common colour scale (dashed line: anchor), then the mean profiles over time
(dotted line: anchor) and frequency.

In [ ]:
extreme_categories = category_order(EXTREME_LEVELS)
figure = plot_category_mean_maps(aggregate.category_mean_maps, aggregate.category_counts, extreme_categories,
                                 grid.frequency_edges_hz, grid.times_s, preprocessing.pre_anchor_s,
                                 f"Mean Grad-CAM map of the predicted class by category: {run_dir.name}")
save_figure(figure, output_dir / "category_mean_maps.png")
save_figure(figure, output_dir / "category_mean_maps.pdf")

In [ ]:
category_labels = categories[CATEGORY_COLUMN]
is_extreme = category_labels.isin(extreme_categories).to_numpy()
time_profiles = mean_profiles_by_category(aggregate.time_profiles[is_extreme], category_labels[is_extreme])
frequency_profiles = mean_profiles_by_category(aggregate.frequency_profiles[is_extreme], category_labels[is_extreme])
figure, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
plot_category_profiles(time_profiles, grid.times_s, "time from window start (s)", False, axes[0],
                       preprocessing.pre_anchor_s)
plot_category_profiles(frequency_profiles, grid.frequencies_hz, "frequency (Hz)", True, axes[1])
figure.suptitle(f"Mean Grad-CAM profiles by category: {run_dir.name}", x=0.01, ha="left")
save_figure(figure, output_dir / "category_profiles.png")
save_figure(figure, output_dir / "category_profiles.pdf")

## 6. Single records

`records_per_category` records drawn from each low- and high-uncertainty category; maps of both classes. One PNG per
record in `explanation/examples/<category>/` and one PDF per category. The first figure is shown below.

In [ ]:
examples = sample_category_records(categories, explanation.records_per_category, explanation.random_seed)
example_records = records.loc[examples["record"].to_numpy()]
example_dataset = FeatureDataset(features_path, representation, example_records[FEATURE_INDEX_COLUMN].to_numpy(),
                                 example_records[columns.label].to_numpy(), statistics)
raw_images, _ = read_features(features_path, representation, example_records[FEATURE_INDEX_COLUMN].to_numpy())
traces = load_waveforms(example_records, config)
preprocessing_grid = grid_from_config(preprocessing)
measure = explanation.uncertainty_measure


def example_figure(position: int):
    """Grad-CAM figure of one drawn record (maps of both classes over MC passes)."""
    example, record = examples.iloc[position], example_records.iloc[position]
    inputs = example_dataset[position][0].unsqueeze(0).to(device)
    maps = grad_cam_by_class(model.to(device), model.last_conv_layer, inputs, explanation.mc_samples)
    window_waveform = process_record(traces[position], record, sampling_rate_hz, preprocessing, preprocessing_grid,
                                     config.waveforms.zero_tolerance).window_waveform
    title = (f"{example[CATEGORY_COLUMN]}: {record['trace_name_old']}, p(bad) {example['probability_mean']:.2f}, "
             f"{measure.replace('_', ' ')} {example[measure]:.3f}")
    return plot_grad_cam(
        raw_images[position], grid.frequencies_hz, grid.frequency_edges_hz, grid.times_s,
        {name: (result.mean[0].cpu().numpy(), result.std[0].cpu().numpy()) for name, result in maps.items()},
        window_waveform, preprocessing.target_sampling_rate_hz, CV_COMPONENTS, title,
    )


figures_by_category = {}
for position in range(len(examples)):
    category = examples.iloc[position][CATEGORY_COLUMN]
    figure = example_figure(position)
    save_figure(figure, output_dir / "examples" / category / f"{example_records.iloc[position]['trace_name_old']}.png")
    figures_by_category.setdefault(category, []).append(figure)
for category, figures in figures_by_category.items():
    save_figures_pdf(figures, output_dir / "examples" / f"{category}.pdf")
print(examples[CATEGORY_COLUMN].value_counts().reindex(extreme_categories, fill_value=0).to_string())
example_figure(0);